# 🧠 MistralMind — Multi-Specialist Fine-Tuning & Routing Agent

**Fine-tune 4 domain specialist LoRA adapters on a single 3B base model, then route queries intelligently.**

| Specialist | Dataset | Domain |
|:---:|:---:|:---:|
| 🏥 Medical | PubMedQA + MedMCQA | Clinical reasoning, evidence-based medicine |
| 📈 Finance | Finance-Alpaca | Quantitative analysis, valuation |
| 💻 Code | CodeAlpaca-20k + Magicoder | Software engineering, algorithms |
| 🎨 Creative | Synthetic-Instruct + WritingPrompts | Creative writing, storytelling |

**Stack:** Unsloth (2x faster QLoRA) + TRL (SFTTrainer) + W&B (experiment tracking) + Llama-3.2-3B-Instruct (base model)

---

> ⚠️ **Runtime:** Go to `Runtime → Change runtime type → T4 GPU` before running.

> 💡 **Tip:** Each specialist trains in ~15–25 min on T4 with 2,000 samples. Run them one at a time to avoid OOM.

## 1️⃣ Install Dependencies

In [ ]:
%%capture
# Install Unsloth (Colab-optimized) + all dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets wandb mistralai gradio python-dotenv pyyaml sentencepiece protobuf

print("✅ All dependencies installed!")

## 2️⃣ GPU Check & Imports

In [ ]:
import torch
import gc
import os
import json
import re
import time
from dataclasses import dataclass
from typing import Optional

# GPU info
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
else:
    raise RuntimeError("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

print(f"🔥 PyTorch {torch.__version__}")
print(f"🔥 CUDA {torch.version.cuda}")

## 3️⃣ Configuration

Set your API keys and training parameters here. All keys are optional — W&B will run in anonymous mode if no key is provided.

In [ ]:
# ============================================================
#  🔑  API KEYS  (set these or leave blank for defaults)
# ============================================================
WANDB_API_KEY   = ""   # https://wandb.ai/authorize  (leave blank for anonymous mode)
HF_TOKEN        = ""   # https://huggingface.co/settings/tokens  (needed for gated models)
MISTRAL_API_KEY = ""   # https://console.mistral.ai  (only for router agent demo)

# ============================================================
#  ⚙️  MODEL & TRAINING CONFIG
# ============================================================
BASE_MODEL      = "unsloth/Llama-3.2-3B-Instruct"  # 3B params, fits T4 easily
MAX_SEQ_LEN     = 2048
LOAD_IN_4BIT    = True

# LoRA hyperparameters (tuned for 3B model on T4)
LORA_RANK       = 32
LORA_ALPHA      = 64
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

# Training parameters (Colab-friendly defaults)
EPOCHS              = 3
BATCH_SIZE          = 2
GRAD_ACCUM_STEPS    = 4        # effective batch = 8
LEARNING_RATE       = 2e-4
WARMUP_RATIO        = 0.05
MAX_SAMPLES         = 2000     # cap per specialist (faster on Colab)
LOGGING_STEPS       = 10
EVAL_STEPS          = 50
SAVE_STEPS          = 100

OUTPUT_DIR          = "/content/checkpoints"

# ============================================================
#  🏥📈💻🎨  SPECIALIST SYSTEM PROMPTS
# ============================================================
SYSTEM_PROMPTS = {
    "medical": (
        "You are a world-class medical expert and clinical reasoning specialist. "
        "You think step by step using evidence-based medicine. You cite relevant studies, "
        "explain pathophysiology clearly, and always prioritize patient safety. "
        "State your confidence level and flag when specialist consultation is needed."
    ),
    "finance": (
        "You are a senior quantitative analyst and financial strategist with deep expertise "
        "in markets, valuation, risk management, and macroeconomics. "
        "Provide rigorous, data-driven analysis with clear reasoning chains. "
        "Always quantify uncertainty and explicitly flag key assumptions."
    ),
    "code": (
        "You are an expert software engineer with mastery across languages and paradigms. "
        "Write clean, efficient, well-documented code. Explain your reasoning thoroughly, "
        "identify edge cases proactively, and follow industry best practices. "
        "Think like a senior engineer conducting a careful code review."
    ),
    "creative": (
        "You are a brilliant creative writer with a distinctive voice and vivid imagination. "
        "Craft engaging narratives, evocative descriptions, and compelling characters. "
        "Use literary techniques masterfully \u2014 show don't tell, subtext, rhythm, metaphor. "
        "Adapt your style fluidly to match any genre, tone, or creative constraint."
    ),
}

print("✅ Configuration loaded")
print(f"   Base model:   {BASE_MODEL}")
print(f"   LoRA rank:    {LORA_RANK} | alpha: {LORA_ALPHA}")
print(f"   Max samples:  {MAX_SAMPLES:,} per specialist")
print(f"   Epochs:       {EPOCHS}")
print(f"   Output:       {OUTPUT_DIR}")

In [ ]:
# ============================================================
#  Initialize W&B
# ============================================================
import wandb

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    print("✅ Logged in to W&B")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("ℹ️  W&B running in offline mode (set WANDB_API_KEY to enable cloud logging)")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ HuggingFace token set")

## 4️⃣ Dataset Loaders

Each specialist loads real HuggingFace datasets and formats them into chat-template conversations.

In [ ]:
from datasets import load_dataset, Dataset

# ============================================================
#  Dataset formatters: raw HF row → {"instruction", "response"}
# ============================================================

def format_pubmedqa(row):
    question = row.get("question", "")
    answer   = row.get("long_answer", "")
    context  = row.get("context", {})
    if not question or not answer or len(answer) < 20:
        return None
    ctx_parts = []
    if isinstance(context, dict):
        for val in context.get("contexts", []):
            ctx_parts.append(str(val))
    ctx = " ".join(ctx_parts[:3])
    instr = f"Research Context:\n{ctx}\n\nClinical Question:\n{question}" if ctx else question
    return {"instruction": instr, "response": answer}

def format_medmcqa(row):
    question = row.get("question", "")
    exp      = row.get("exp", "")
    if not question:
        return None
    options = ""
    for i, key in enumerate(["opa", "opb", "opc", "opd"]):
        opt = row.get(key, "")
        if opt:
            options += f"\n{chr(65+i)}. {opt}"
    user = f"{question}{options}" if options else question
    cop_map = {0: "opa", 1: "opb", 2: "opc", 3: "opd"}
    cop = row.get("cop", -1)
    parts = []
    if cop in cop_map:
        c = row.get(cop_map[cop], "")
        if c:
            parts.append(f"The correct answer is {chr(65+cop)}. {c}")
    if exp:
        parts.append(f"Explanation: {exp}")
    answer = "\n\n".join(parts) if parts else exp or "No explanation."
    if len(answer) < 10:
        return None
    return {"instruction": user, "response": answer}

def format_finance_alpaca(row):
    instr  = row.get("instruction", "").strip()
    inp    = row.get("input", "").strip()
    output = row.get("output", "").strip()
    if not instr or not output or len(output) < 10:
        return None
    if inp:
        instr = f"{instr}\n\nContext: {inp}"
    return {"instruction": instr, "response": output}

def format_code_alpaca(row):
    instr  = (row.get("instruction", "") or row.get("prompt", "")).strip()
    inp    = row.get("input", "").strip()
    output = (row.get("output", "") or row.get("completion", "")).strip()
    if not instr or not output or len(output) < 10:
        return None
    if inp:
        instr = f"{instr}\n\n```\n{inp}\n```"
    return {"instruction": instr, "response": output}

def format_magicoder(row):
    problem  = row.get("problem", "").strip()
    solution = row.get("solution", "").strip()
    if not problem or not solution or len(solution) < 10:
        return None
    return {"instruction": problem, "response": solution}

def format_creative_pairwise(row):
    prompt = row.get("prompt", "").strip()
    chosen = row.get("chosen", "").strip()
    if not prompt or not chosen or len(chosen) < 20:
        return None
    prompt = prompt.replace("Human:", "").replace("Assistant:", "").strip()
    return {"instruction": prompt, "response": chosen}

def format_writingprompts(row):
    prompt = row.get("prompt", "").strip()
    story  = row.get("story", "").strip()
    if not prompt or not story or len(story) < 50:
        return None
    prompt = prompt.removeprefix("[WP]").removeprefix("[SP]").strip()
    if len(story) > 3000:
        story = story[:3000] + "..."
    return {"instruction": f"Write a story: {prompt}", "response": story}

print("✅ Formatters defined")

In [ ]:
# ============================================================
#  Master dataset loader per specialist
# ============================================================

DATASET_SOURCES = {
    "medical": [
        {"id": "qiaojin/PubMedQA", "name": "pqa_labeled", "split": "train", "formatter": format_pubmedqa},
        {"id": "openlifescienceai/medmcqa", "name": None, "split": "train", "formatter": format_medmcqa},
    ],
    "finance": [
        {"id": "gbharti/finance-alpaca", "name": None, "split": "train", "formatter": format_finance_alpaca},
    ],
    "code": [
        {"id": "sahil2801/CodeAlpaca-20k", "name": None, "split": "train", "formatter": format_code_alpaca},
        {"id": "ise-uiuc/Magicoder-OSS-Instruct-75K", "name": None, "split": "train", "formatter": format_magicoder},
    ],
    "creative": [
        {"id": "Dahoas/synthetic-instruct-gptj-pairwise", "name": None, "split": "train", "formatter": format_creative_pairwise},
    ],
}


def load_specialist_data(specialist: str, max_samples: int = MAX_SAMPLES):
    """Load, format, and combine all datasets for a specialist."""
    sources = DATASET_SOURCES[specialist]
    all_formatted = []

    for src in sources:
        print(f"  \U0001f4e5 Loading {src['id']}...")
        try:
            kwargs = {"path": src["id"], "split": src["split"], "trust_remote_code": True}
            if src["name"]:
                kwargs["name"] = src["name"]
            ds = load_dataset(**kwargs)
            for row in ds:
                result = src["formatter"](row)
                if result:
                    all_formatted.append(result)
            print(f"    ✔ Got {len(all_formatted)} samples so far")
        except Exception as e:
            print(f"    ⚠️ Could not load {src['id']}: {e}")

    # Shuffle and cap
    import random
    random.seed(42)
    random.shuffle(all_formatted)
    all_formatted = all_formatted[:max_samples]

    ds = Dataset.from_list(all_formatted)
    print(f"  ✅ {specialist}: {len(ds):,} samples ready")
    return ds

print("✅ Dataset loader ready")

## 5️⃣ Load Base Model (Llama-3.2-3B-Instruct)

We load the base model **once** and reuse it for all specialists. LoRA adapters are swapped per specialist.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print(f"\n\U0001f527 Loading base model: {BASE_MODEL}")
print(f"   Seq length: {MAX_SEQ_LEN} | 4-bit: {LOAD_IN_4BIT}\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,         # auto-detect bf16/fp16
    load_in_4bit   = LOAD_IN_4BIT,
)

# Apply Llama 3.1 chat template (compatible with 3.2)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

print(f"\n\u2705 Model loaded: {model.config._name_or_path}")
print(f"   Parameters:  {sum(p.numel() for p in model.parameters()):,}")
print(f"   GPU Memory:  {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 6️⃣ Training Pipeline

The training function handles:
1. LoRA adapter creation
2. Dataset loading & chat-template formatting
3. SFT training with W&B logging
4. Checkpoint saving

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer


def build_chat_formatter(tok, system_prompt):
    """Convert {instruction, response} → chat-template text."""
    def fn(example):
        messages = [
            {"role": "system",    "content": system_prompt},
            {"role": "user",      "content": example["instruction"]},
            {"role": "assistant", "content": example["response"]},
        ]
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        return {"text": text}
    return fn


def train_specialist(specialist_name: str):
    """
    Full training pipeline for one specialist.
    Reloads the base model fresh each time to avoid LoRA conflicts.
    """
    global model, tokenizer

    print(f"\n{'='*60}")
    print(f"  \U0001f680 Training [{specialist_name.upper()}] Specialist")
    print(f"{'='*60}\n")

    system_prompt = SYSTEM_PROMPTS[specialist_name]
    output_path   = os.path.join(OUTPUT_DIR, specialist_name)

    # \u2500\u2500 Reload base model fresh \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = BASE_MODEL,
        max_seq_length = MAX_SEQ_LEN,
        dtype          = None,
        load_in_4bit   = LOAD_IN_4BIT,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

    # \u2500\u2500 LoRA adapter \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    model = FastLanguageModel.get_peft_model(
        model,
        r                          = LORA_RANK,
        target_modules             = TARGET_MODULES,
        lora_alpha                 = LORA_ALPHA,
        lora_dropout               = LORA_DROPOUT,
        bias                       = "none",
        use_gradient_checkpointing = "unsloth",
        random_state               = 42,
        use_rslora                 = True,
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    # \u2500\u2500 Dataset \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    print(f"\n\U0001f4da Loading dataset for [{specialist_name}]...")
    dataset   = load_specialist_data(specialist_name)
    formatter = build_chat_formatter(tokenizer, system_prompt)
    dataset   = dataset.map(formatter, remove_columns=dataset.column_names)

    split   = dataset.train_test_split(test_size=0.05, seed=42)
    train_d = split["train"]
    eval_d  = split["test"]
    print(f"   Train: {len(train_d):,} | Eval: {len(eval_d):,}")

    # \u2500\u2500 W&B \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    wandb.init(
        project = "mistralmind",
        name    = f"{specialist_name}-specialist-v1",
        tags    = ["mistralmind", specialist_name, "unsloth", "qlora", "3B"],
        config  = {
            "specialist":     specialist_name,
            "base_model":     BASE_MODEL,
            "epochs":         EPOCHS,
            "max_seq_len":    MAX_SEQ_LEN,
            "lora_rank":      LORA_RANK,
            "lora_alpha":     LORA_ALPHA,
            "learning_rate":  LEARNING_RATE,
            "max_samples":    MAX_SAMPLES,
            "train_samples":  len(train_d),
            "eval_samples":   len(eval_d),
        },
    )

    # Log sample data
    sample_table = wandb.Table(
        columns=["text"],
        data=[[train_d[i]["text"]] for i in range(min(3, len(train_d)))]
    )
    wandb.log({"training_samples": sample_table})

    # \u2500\u2500 Training \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    training_args = TrainingArguments(
        output_dir                  = output_path,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM_STEPS,
        warmup_ratio                = WARMUP_RATIO,
        learning_rate               = LEARNING_RATE,
        fp16            = not torch.cuda.is_bf16_supported(),
        bf16            = torch.cuda.is_bf16_supported(),
        logging_steps   = LOGGING_STEPS,
        eval_strategy   = "steps",
        eval_steps      = EVAL_STEPS,
        save_strategy   = "steps",
        save_steps      = SAVE_STEPS,
        save_total_limit       = 2,
        load_best_model_at_end = True,
        metric_for_best_model  = "eval_loss",
        report_to              = "wandb",
        optim                  = "adamw_8bit",
        lr_scheduler_type      = "cosine",
        seed                   = 42,
        group_by_length        = True,
    )

    trainer = SFTTrainer(
        model              = model,
        tokenizer          = tokenizer,
        train_dataset      = train_d,
        eval_dataset       = eval_d,
        dataset_text_field = "text",
        max_seq_length     = MAX_SEQ_LEN,
        args               = training_args,
        packing            = True,
    )

    print(f"\n\U0001f680 Training [{specialist_name}] \u2014 {EPOCHS} epochs...")
    trainer_stats = trainer.train()

    # \u2500\u2500 Log final metrics \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    wandb.log({
        "final_train_loss":      trainer_stats.training_loss,
        "total_steps":           trainer_stats.global_step,
        "train_runtime_seconds": trainer_stats.metrics["train_runtime"],
        "samples_per_second":    trainer_stats.metrics["train_samples_per_second"],
    })

    # \u2500\u2500 Save LoRA adapter \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    lora_path = os.path.join(output_path, "lora_adapter")
    model.save_pretrained(lora_path)
    tokenizer.save_pretrained(lora_path)
    print(f"   \U0001f4be LoRA adapter saved \u2192 {lora_path}")

    # \u2500\u2500 Save merged 16-bit (optional, for deployment) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    merged_path = os.path.join(output_path, "merged_16bit")
    model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")
    print(f"   \U0001f4be Merged model saved \u2192 {merged_path}")

    print(f"\n\u2705 [{specialist_name.upper()}] training complete!")
    print(f"   Final loss: {trainer_stats.training_loss:.4f}")
    print(f"   Steps:      {trainer_stats.global_step}")
    print(f"   Runtime:    {trainer_stats.metrics['train_runtime']:.0f}s")

    wandb.finish()
    return lora_path


print("✅ Training pipeline ready")

## 7️⃣ Train All Specialists

Run each specialist one at a time. Each cell reloads the base model fresh, applies a new LoRA, trains, and saves.

> 💡 You can skip any specialist by not running its cell.

### 🏥 Train Medical Specialist

In [ ]:
medical_lora = train_specialist("medical")

### 📈 Train Finance Specialist

In [ ]:
finance_lora = train_specialist("finance")

### 💻 Train Code Specialist

In [ ]:
code_lora = train_specialist("code")

### 🎨 Train Creative Specialist

In [ ]:
creative_lora = train_specialist("creative")

## 8️⃣ Evaluation — Before vs After Fine-Tuning

Benchmark each specialist with domain-specific metrics:
- **Medical**: Clinical keyword coverage + safety score
- **Finance**: Numerical calculation accuracy
- **Code**: Pass@1 on coding tasks (exec-based)
- **Creative**: Vocabulary richness + sensory language + specificity

In [ ]:
# ============================================================
#  Evaluation Helper
# ============================================================

@dataclass
class EvalResult:
    specialist:  str
    phase:       str
    metric_name: str
    score:       float
    details:     dict


def generate_response(mdl, tok, system: str, user: str,
                      temperature: float = 0.3, max_new: int = 512) -> str:
    """Generate a response from the model."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    inputs = tok.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(mdl.device)

    with torch.no_grad():
        output = mdl.generate(
            inputs,
            max_new_tokens = max_new,
            temperature    = temperature,
            do_sample      = temperature > 0,
            pad_token_id   = tok.eos_token_id,
        )
    return tok.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)


print("✅ Evaluation helpers ready")

In [ ]:
# ============================================================
#  🏥 Medical Evaluation Benchmark
# ============================================================

MEDICAL_BENCH = [
    {
        "q": ("A 45-year-old male presents with sudden crushing chest pain radiating to the left arm, "
              "diaphoresis, and nausea for 30 minutes. BP 140/90, HR 105. Most likely diagnosis "
              "and immediate management steps?"),
        "keywords": ["myocardial infarction", "MI", "aspirin", "nitroglycerin", "ECG", "troponin"],
        "safety":   ["emergency", "immediate", "urgent", "hospital"],
    },
    {
        "q": ("Explain the mechanism by which metformin reduces blood glucose in type 2 diabetes, "
              "and list its major contraindications."),
        "keywords": ["AMPK", "gluconeogenesis", "hepatic", "insulin", "lactic acidosis"],
        "safety":   ["contraindicated", "avoid", "renal"],
    },
    {
        "q": ("A child presents with fever, neck stiffness, photophobia, and a petechial rash. "
              "What is the most urgent diagnosis and immediate treatment?"),
        "keywords": ["meningococcal", "meningitis", "ceftriaxone", "penicillin"],
        "safety":   ["emergency", "immediate", "antibiotic"],
    },
]


def evaluate_medical(mdl, tok, phase):
    sys_p = SYSTEM_PROMPTS["medical"]
    results, scores = [], []
    for i, item in enumerate(MEDICAL_BENCH):
        resp = generate_response(mdl, tok, sys_p, item["q"]).lower()
        kw_score = sum(1 for k in item["keywords"] if k.lower() in resp) / len(item["keywords"])
        sf_score = sum(1 for k in item["safety"] if k.lower() in resp) / len(item["safety"]) if item["safety"] else 1.0
        combined = 0.65 * kw_score + 0.35 * sf_score
        scores.append(combined)
        results.append(EvalResult("medical", phase, f"q{i+1}", combined,
                                  {"kw": round(kw_score, 3), "safety": round(sf_score, 3)}))
    avg = sum(scores) / len(scores)
    results.append(EvalResult("medical", phase, "overall", avg, {}))
    return results

print("✅ Medical benchmark defined")

In [ ]:
# ============================================================
#  📈 Finance Evaluation Benchmark
# ============================================================

FINANCE_BENCH = [
    {
        "q": ("A company has $500M revenue, EBITDA margin 20%, sector EV/EBITDA 12x, "
              "$50M net debt. Calculate equity value."),
        "expected": 1150.0,
        "tolerance": 0.05,
    },
    {
        "q": "Present value of $1,000 received in 5 years at 8% discount rate?",
        "expected": 680.58,
        "tolerance": 0.03,
    },
    {
        "q": "Portfolio: 12% annual return, 4% risk-free rate, 15% std dev. Sharpe ratio?",
        "expected": 0.533,
        "tolerance": 0.05,
    },
]


def extract_number(text):
    patterns = [
        r'(?:value|answer|result|price|ratio)\s*(?:is|=|:)\s*\$?\s*([\d,]+\.?\d*)',
        r'\$\s*([\d,]+\.?\d*)', r'\u2248\s*([\d,]+\.?\d*)',
        r'=\s*([\d,]+\.?\d*)', r'([\d,]+\.\d{2,})',
    ]
    for p in patterns:
        for m in re.finditer(p, text, re.IGNORECASE):
            try:
                v = float(m.group(1).replace(',', ''))
                if v > 0:
                    return v
            except:
                continue
    return None


def evaluate_finance(mdl, tok, phase):
    sys_p = SYSTEM_PROMPTS["finance"]
    results, scores = [], []
    for i, item in enumerate(FINANCE_BENCH):
        resp = generate_response(mdl, tok, sys_p, item["q"], temperature=0.1)
        num  = extract_number(resp)
        if num is not None:
            err   = abs(num - item["expected"]) / abs(item["expected"])
            score = max(0.0, 1.0 - err / item["tolerance"])
        else:
            score = 0.0
        scores.append(score)
        results.append(EvalResult("finance", phase, f"q{i+1}", score,
                                  {"expected": item["expected"], "got": num}))
    avg = sum(scores) / len(scores)
    results.append(EvalResult("finance", phase, "overall", avg, {}))
    return results

print("✅ Finance benchmark defined")

In [ ]:
# ============================================================
#  💻 Code Evaluation Benchmark (exec-based pass@1)
# ============================================================

CODE_BENCH = [
    {
        "prompt": "Write a Python function `fibonacci(n)` returning the nth Fibonacci number (0-indexed). fibonacci(0)=0, fibonacci(1)=1.",
        "tests": [("fibonacci(0)", 0), ("fibonacci(1)", 1), ("fibonacci(10)", 55)],
    },
    {
        "prompt": "Write `two_sum(nums, target)` returning indices of two numbers summing to target. Use a hash map.",
        "tests": [("sorted(two_sum([2,7,11,15], 9))", [0,1]), ("sorted(two_sum([3,2,4], 6))", [1,2])],
    },
    {
        "prompt": "Write `is_palindrome(s)` that checks if s is a palindrome, ignoring spaces and case.",
        "tests": [("is_palindrome('racecar')", True), ("is_palindrome('hello')", False)],
    },
]


def extract_code(text):
    m = re.search(r'```(?:python)?\n(.*?)```', text, re.DOTALL)
    return m.group(1).strip() if m else text.strip()


def evaluate_code(mdl, tok, phase):
    code_sys = "You are an expert Python developer. Write only the requested function in a ```python``` block."
    results = []
    total_pass, total_tests = 0, 0
    for i, task in enumerate(CODE_BENCH):
        resp = generate_response(mdl, tok, code_sys, task["prompt"], temperature=0.2, max_new=600)
        code = extract_code(resp)
        ns, passed = {}, 0
        try:
            exec(code, ns)
            for expr, expected in task["tests"]:
                try:
                    if eval(expr, ns) == expected:
                        passed += 1
                except:
                    pass
        except:
            pass
        score = passed / len(task["tests"])
        total_pass  += passed
        total_tests += len(task["tests"])
        results.append(EvalResult("code", phase, f"task{i+1}", score,
                                  {"passed": passed, "total": len(task["tests"])}))
    overall = total_pass / total_tests if total_tests else 0
    results.append(EvalResult("code", phase, "overall_pass@1", overall, {}))
    return results

print("✅ Code benchmark defined")

In [ ]:
# ============================================================
#  🎨 Creative Evaluation Benchmark
# ============================================================

CREATIVE_BENCH = [
    "Write the opening paragraph of a noir detective story in a cyberpunk city.",
    "Write a haiku about receiving a message from someone you've missed.",
    "A scientist discovers their life's work was wrong. Describe the moment in 3 sentences.",
]

SENSORY_WORDS = [
    "glow", "shadow", "whisper", "silence", "gleam", "haze", "flicker", "hum",
    "neon", "smoke", "cold", "warm", "sharp", "bitter", "echo", "fade", "pulse",
]


def score_creative(text):
    words  = text.lower().split()
    unique = set(words)
    ttr = len(unique) / len(words) if words else 0
    sensory = min(1.0, sum(1 for w in SENSORY_WORDS if w in text.lower()) / 4)
    wc = len(words)
    length_score = min(1.0, wc / 20) if wc < 20 else (1.0 if wc <= 200 else max(0.5, 1.0 - (wc-200)/400))
    combined = 0.4 * ttr + 0.35 * sensory + 0.25 * length_score
    return combined


def evaluate_creative(mdl, tok, phase):
    sys_p = SYSTEM_PROMPTS["creative"]
    results, scores = [], []
    for i, prompt in enumerate(CREATIVE_BENCH):
        resp = generate_response(mdl, tok, sys_p, prompt, temperature=0.85, max_new=300)
        score = score_creative(resp)
        scores.append(score)
        results.append(EvalResult("creative", phase, f"prompt{i+1}", score,
                                  {"preview": resp[:100]}))
    avg = sum(scores) / len(scores)
    results.append(EvalResult("creative", phase, "overall", avg, {}))
    return results


EVALUATORS = {
    "medical":  evaluate_medical,
    "finance":  evaluate_finance,
    "code":     evaluate_code,
    "creative": evaluate_creative,
}

print("✅ Creative benchmark defined")

In [ ]:
# ============================================================
#  Run full before/after evaluation for all specialists
# ============================================================

def run_full_evaluation(specialists=None):
    """Evaluate base model (before) vs fine-tuned (after) for each specialist."""
    global model, tokenizer

    if specialists is None:
        specialists = ["medical", "finance", "code", "creative"]

    all_results = {}

    for spec in specialists:
        print(f"\n{'='*60}")
        print(f"  \U0001f50d Evaluating [{spec.upper()}]")
        print(f"{'='*60}")

        evaluator = EVALUATORS[spec]
        checkpoint = os.path.join(OUTPUT_DIR, spec, "lora_adapter")

        # \u2500\u2500 BEFORE (base model) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
        print(f"\n  \u23f3 Phase: BEFORE (base model)")
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN,
            dtype=None, load_in_4bit=LOAD_IN_4BIT,
        )
        tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
        FastLanguageModel.for_inference(model)

        before_results = evaluator(model, tokenizer, "before")

        # \u2500\u2500 AFTER (fine-tuned) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
        if os.path.exists(checkpoint):
            print(f"  \u23f3 Phase: AFTER (fine-tuned)")
            del model, tokenizer
            gc.collect()
            torch.cuda.empty_cache()

            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=checkpoint, max_seq_length=MAX_SEQ_LEN,
                dtype=None, load_in_4bit=LOAD_IN_4BIT,
            )
            tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
            FastLanguageModel.for_inference(model)

            after_results = evaluator(model, tokenizer, "after")
        else:
            print(f"  \u26a0\ufe0f  No checkpoint found at {checkpoint}, skipping AFTER phase")
            after_results = []

        all_results[spec] = {"before": before_results, "after": after_results}

        # \u2500\u2500 Print comparison \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
        print(f"\n  {'Metric':<25} {'Before':>8} {'After':>8} {'\u0394':>8}")
        print(f"  {'\u2500'*51}")
        for br in before_results:
            ar = next((a for a in after_results if a.metric_name == br.metric_name), None)
            after_val = ar.score if ar else 0.0
            delta     = after_val - br.score
            arrow     = "\u2191" if delta > 0 else ("\u2193" if delta < 0 else "=")
            print(f"  {br.metric_name:<25} {br.score:>8.3f} {after_val:>8.3f} {arrow}{abs(delta):>7.3f}")

    return all_results

print("✅ Evaluation runner ready")

### Run Evaluation

This compares the **base Llama-3.2-3B** against each **fine-tuned specialist** on domain-specific benchmarks.

In [ ]:
# Run evaluation for all trained specialists
# Change the list to evaluate only specific specialists
eval_results = run_full_evaluation(["medical", "finance", "code", "creative"])

## 9️⃣ Test Inference — Interactive Specialist Demo

Load a fine-tuned specialist and ask it questions to see the improvement.

In [ ]:
# ============================================================
#  Quick inference test with a fine-tuned specialist
# ============================================================

def test_specialist(specialist_name: str, question: str):
    """Load a fine-tuned specialist and generate a response."""
    global model, tokenizer

    checkpoint = os.path.join(OUTPUT_DIR, specialist_name, "lora_adapter")
    if not os.path.exists(checkpoint):
        print(f"\u274c No checkpoint found for {specialist_name}!")
        return

    # Reload with fine-tuned adapter
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=checkpoint, max_seq_length=MAX_SEQ_LEN,
        dtype=None, load_in_4bit=LOAD_IN_4BIT,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
    FastLanguageModel.for_inference(model)

    system = SYSTEM_PROMPTS[specialist_name]
    response = generate_response(model, tokenizer, system, question,
                                  temperature=0.4, max_new=1024)

    print(f"\n{'='*60}")
    print(f"  \U0001f9e0 [{specialist_name.upper()}] Specialist")
    print(f"{'='*60}")
    print(f"\n\U0001f4ac Question: {question}\n")
    print(f"\U0001f916 Response:\n{response}")
    return response

In [ ]:
# 🏥 Test Medical Specialist
test_specialist("medical",
    "A 55-year-old diabetic patient presents with a non-healing foot ulcer. "
    "What are the key assessment steps and management priorities?"
)

In [ ]:
# 📈 Test Finance Specialist
test_specialist("finance",
    "A startup has projected revenues of $2M in year 1, growing 40% annually for 5 years. "
    "Using a 12% discount rate, what is the present value of these future cash flows?"
)

In [ ]:
# 💻 Test Code Specialist
test_specialist("code",
    "Write a Python class for a thread-safe LRU cache with get() and put() methods, "
    "both O(1). Include type hints and docstrings."
)

In [ ]:
# 🎨 Test Creative Specialist
test_specialist("creative",
    "Write a short scene where two time travelers meet at a coffee shop, "
    "each unaware the other is also from a different era."
)

## 💾 Save & Download Models

Download your trained LoRA adapters from Colab, or push them to HuggingFace Hub.

In [ ]:
# ============================================================
#  List all saved checkpoints
# ============================================================
import subprocess

print("\U0001f4c1 Saved checkpoints:\n")
for spec in ["medical", "finance", "code", "creative"]:
    lora_path = os.path.join(OUTPUT_DIR, spec, "lora_adapter")
    if os.path.exists(lora_path):
        size = subprocess.getoutput(f"du -sh {lora_path}")
        print(f"  \u2705 {spec:<10} {lora_path}  ({size.split()[0]})")
    else:
        print(f"  \u274c {spec:<10} not trained yet")

In [ ]:
# ============================================================
#  Download as ZIP (for Colab)
# ============================================================
import shutil

# Zip all LoRA adapters
zip_path = "/content/mistralmind_lora_adapters"
if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")

adapters_to_zip = {}
for spec in ["medical", "finance", "code", "creative"]:
    lora_path = os.path.join(OUTPUT_DIR, spec, "lora_adapter")
    if os.path.exists(lora_path):
        adapters_to_zip[spec] = lora_path

if adapters_to_zip:
    # Create temp directory with all adapters
    tmp_dir = "/content/_tmp_adapters"
    os.makedirs(tmp_dir, exist_ok=True)
    for spec, path in adapters_to_zip.items():
        shutil.copytree(path, os.path.join(tmp_dir, spec), dirs_exist_ok=True)
    shutil.make_archive(zip_path, 'zip', tmp_dir)
    shutil.rmtree(tmp_dir)
    print(f"\u2705 ZIP created: {zip_path}.zip")
    print(f"   Download from the Files panel (left sidebar) or run:")
    print(f"   from google.colab import files; files.download('{zip_path}.zip')")
else:
    print("\u274c No adapters found. Train specialists first!")

In [ ]:
# ============================================================
#  Push to HuggingFace Hub (optional)
# ============================================================

HF_USERNAME = ""  # Your HuggingFace username

if HF_TOKEN and HF_USERNAME:
    for spec in ["medical", "finance", "code", "creative"]:
        lora_path = os.path.join(OUTPUT_DIR, spec, "lora_adapter")
        if os.path.exists(lora_path):
            # Reload model with adapter
            del model, tokenizer
            gc.collect()
            torch.cuda.empty_cache()

            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=lora_path, max_seq_length=MAX_SEQ_LEN,
                dtype=None, load_in_4bit=LOAD_IN_4BIT,
            )
            repo_id = f"{HF_USERNAME}/mistralmind-{spec}-3b"
            model.push_to_hub_merged(
                repo_id, tokenizer,
                save_method="lora", token=HF_TOKEN,
            )
            print(f"  \u2705 Pushed {spec} \u2192 https://huggingface.co/{repo_id}")
else:
    print("\u2139\ufe0f  Set HF_TOKEN and HF_USERNAME above to push models to HuggingFace Hub")

---

## \u2705 Done!

**What you built:**
1. \U0001f527 Loaded Llama-3.2-3B-Instruct as a 4-bit quantized base model
2. \U0001f3cb\ufe0f Fine-tuned 4 domain specialists with QLoRA (medical, finance, code, creative)
3. \U0001f4ca Benchmarked before/after performance with domain-specific metrics
4. \U0001f9ea Tested each specialist with interactive inference
5. \U0001f4be Saved LoRA adapters (~50MB each vs 6GB full model)

**Next steps:**
- Download the LoRA adapters and use them with the routing agent (`agent/router.py`)
- Run the Gradio demo (`demo/app.py`) locally or on HuggingFace Spaces
- Push adapters to HuggingFace Hub for sharing